In [37]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
    NoSuchElementException
)
import time
import csv

def scrape_mtj_fragrances(output_csv="page3.csv"):
    # --- 1. Launch Chrome in headless mode ------------------------------------
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")                    # no GUI
    driver = webdriver.Chrome(options=options)
    
    # --- 2. Navigate to the Fragrances collection ---------------------------
    # driver.get("https://mtjonline.com/collections/fragrances")
    driver.get("https://mtjonline.com/collections/fragrances?page=3")  # Updated to scrape all products
    wait = WebDriverWait(driver, 15)
    
    # --- 3. Click “Show More” over and over until it disappears -------------
    while True:
        try:
            # a) Wait for the button to be clickable
            btn = wait.until(EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), 'show more')]"
            )))
            # b) Scroll into view (some themes lazy-load)
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.3)                              # let any animation finish
            btn.click()                                  # load more products
            
            # c) Wait briefly for the old button element to go stale
            try:
                wait.until(EC.staleness_of(btn))
            except TimeoutException:
                pass
            time.sleep(1)                                # ensure new items appear
        except (TimeoutException, StaleElementReferenceException, NoSuchElementException):
            # no more “Show More” → exit loop
            break

    # --- 4. Gather all product links on the fully-loaded page ----------------
    products = []
    seen_urls = set()
    
    # We pull every <a> whose href contains “/products/”
    anchors = driver.find_elements(
        By.XPATH,
        "//a[contains(@href, '/products/')]"
    )
    
    for a in anchors:
        try:
            url = a.get_attribute("href").strip()
            if not url or url in seen_urls:
                continue

            # --- 4a. Get the title ---------------------------------------------
            #  1. link’s title="" attribute
            #  2. link’s visible text
            #  3. fallback to the image alt=""
            title = a.get_attribute("title") or a.text.strip()
            if not title:
                try:
                    img = a.find_element(By.TAG_NAME, "img")
                    title = img.get_attribute("alt").strip()
                except NoSuchElementException:
                    title = "N/A"

            # --- 4b. Get the price ---------------------------------------------
            # Look for the next element in the DOM containing “PKR”
            try:
                price_el = a.find_element(
                    By.XPATH,
                    "following::*[contains(text(),'PKR')][1]"
                )
                price = price_el.text.strip()
            except NoSuchElementException:
                price = "N/A"

            # --- 4c. Record and dedupe -----------------------------------------
            products.append({
                "title": title,
                "price": price,
                "url": url
            })
            seen_urls.add(url)

        except Exception:
            # any unexpected per-link error → skip it
            continue

    # --- 5. Tear down WebDriver -----------------------------------------------
    driver.quit()

    # --- 6. Write results to CSV ---------------------------------------------
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "price", "url"])
        writer.writeheader()
        writer.writerows(products)

    print(f"Scraped {len(products)} unique products — saved to '{output_csv}'.")

if __name__ == "__main__":
    scrape_mtj_fragrances()


Scraped 24 unique products — saved to 'page3.csv'.
